In [59]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

In [2]:
BUCKET = "ads508-team1-sephora"
S3_FINAL_SPLITS = "Model/final_splits"

def load_split(name):
    return pd.read_parquet(f"s3://{BUCKET}/{S3_FINAL_SPLITS}/{name}.parquet")

# Product
X_train_product = load_split("product_X_train")
X_val_product   = load_split("product_X_val")
X_test_product  = load_split("product_X_test")
y_train_product = load_split("product_y_train")["rating"]
y_val_product   = load_split("product_y_val")["rating"]
y_test_product  = load_split("product_y_test")["rating"]

# Review
X_train_review = load_split("review_X_train")
X_val_review   = load_split("review_X_val")
X_test_review  = load_split("review_X_test")
y_train_review = load_split("review_y_train")["rating"]
y_val_review   = load_split("review_y_val")["rating"]
y_test_review  = load_split("review_y_test")["rating"]

# Product Modeling

In [47]:
# Remove 'primary_category' column which only include 'Makeup'
cols_to_drop = ["primary_category"]

X_train_product = X_train_product.drop(columns=cols_to_drop)
X_val_product   = X_val_product.drop(columns=cols_to_drop)
X_test_product  = X_test_product.drop(columns=cols_to_drop)

In [51]:
# y_train_product has NaN; need to remove those instances along with indices in X

# training
train_mask = y_train_product.notna()
X_train_product = X_train_product.loc[train_mask].copy()
y_train_product = y_train_product.loc[train_mask].copy()

# validation
val_mask = y_val_product.notna()
X_val_product = X_val_product.loc[val_mask].copy()
y_val_product = y_val_product.loc[val_mask].copy()

# test
test_mask = y_test_product.notna()
X_test_product = X_test_product.loc[test_mask].copy()
y_test_product = y_test_product.loc[test_mask].copy()

In [63]:
X_train_product.head()

,limited_edition,new,online_only,out_of_stock,sephora_exclusive,child_count,child_max_price,child_min_price,price_log,is_on_sale,...,ing_vp/hexadecene copolymer,ing_water,ing_xanthan gum,ing_yellow 5 lake (ci 19140),ing_yellow 5 lake (ci 19140).,ing_yellow 5 lake),ing_yellow 6 lake (ci 15985),ing_zea mays (corn) starch,ing_zinc myristate,ing_zinc stearate
0,0,0,0,0,0,-0.512608,-0.900507,-0.877163,0.318144,0,...,0.0,0.000000,0.272725,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
1,0,0,0,0,0,-0.512608,-0.900507,-0.877163,-1.376052,0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,0,0,0,0,1,3.168607,0.177735,0.246876,-0.787481,0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.342301
3,0,0,0,0,0,3.276878,1.795097,1.932936,1.132988,0,...,0.0,0.155669,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
4,0,0,0,0,1,-0.512608,-0.900507,-0.877163,1.520518,0,...,0.0,0.000000,0.000000,0.0,0.022031,0.0,0.0,0.0,0.0,0.000000


## 1) Linear Regression

In [52]:
lr = LinearRegression()
lr.fit(X_train_product, y_train_product)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [55]:
y_val_pred = lr.predict(X_val_product)

mse = mean_squared_error(y_val_product, y_val_pred)
rmse = np.sqrt(mse)

r2 = r2_score(y_val_product, y_val_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

RMSE: 0.5247
R2: -0.0330


In [57]:
# Add constant (intercept)
X_train_sm = sm.add_constant(X_train_product)

# Fit model
model = sm.OLS(y_train_product, X_train_sm).fit()

# Show summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 rating   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.143
Method:                 Least Squares   F-statistic:                     2.245
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           3.66e-23
Time:                        02:22:01   Log-Likelihood:                -1273.4
No. Observations:                2099   AIC:                             3113.
Df Residuals:                    1816   BIC:                             4711.
Df Model:                         282                                         
Covariance Type:            nonrobust                                         
                                                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

## 2) Random Forest

In [58]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_product, y_train_product)

,n_estimators,300
,criterion,'squared_error'
,max_depth,15
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [61]:
y_val_pred_rf = rf.predict(X_val_product)

rmse_rf = np.sqrt(mean_squared_error(y_val_product, y_val_pred_rf))
r2_rf = r2_score(y_val_product, y_val_pred_rf)

print(f"RF RMSE: {rmse_rf:.4f}")
print(f"RF R2: {r2_rf:.4f}")

RF RMSE: 0.4854
RF R2: 0.1159


In [66]:
# Feature importance
feature_importance = pd.DataFrame({
    "feature": X_train_product.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance.head(20)

,feature,importance
8,price_log,0.104301
277,ing_water,0.038538
12,num_ingredients,0.031382
10,discount_pct,0.023676
73,brand_group_Hourglass,0.021576
2,online_only,0.018205
209,ing_palmitic acid,0.017497
269,ing_trimethylsiloxysilicate,0.015020
39,tertiary_category_Face Brushes,0.014484
230,ing_potassium sorbate,0.013677


## 3) XGBoost

In [60]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train_product, y_train_product)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# Review Modeling

## 1) Logistic Regression

In [3]:
# Hyperparameter Tuning
results = []

for C in [0.1, 1]:
    for class_weight in [None, "balanced"]:
        model = LogisticRegression(
            C=C,
            penalty="l2",
            solver="saga",
            max_iter=2000,
            n_jobs=-1
        )
        
        model.fit(X_train_review, y_train_review)
        preds = model.predict(X_val_review)
        
        results.append({
            "C": C,
            "class_weight": class_weight,
            "val_accuracy": accuracy_score(y_val_review, preds),
            "val_f1_macro": f1_score(y_val_review, preds, average="macro")
        })

results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False)
print(results_df)

     C class_weight  val_accuracy  val_f1_macro
3  1.0     balanced         0.712      0.474184
2  1.0         None         0.712      0.474184
1  0.1     balanced         0.664      0.305247
0  0.1         None         0.664      0.305247


In [4]:
best_log_reg = LogisticRegression(
    C=1,
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    n_jobs=-1
)

best_log_reg.fit(X_train_review, y_train_review)

val_preds = best_log_reg.predict(X_val_review)

print(classification_report(y_val_review, val_preds))

              precision    recall  f1-score   support

           1       0.60      0.56      0.58        27
           2       0.56      0.16      0.24        32
           3       0.38      0.24      0.30        33
           4       0.54      0.34      0.42        95
           5       0.77      0.95      0.85       313

    accuracy                           0.71       500
   macro avg       0.57      0.45      0.48       500
weighted avg       0.68      0.71      0.68       500



In [16]:
feature_names = X_train_review.columns

coef = best_log_reg.coef_
classes = best_log_reg.classes_

print("Classes:", classes)

Classes: [1 2 3 4 5]


In [17]:
def get_top_words(model, feature_names, class_label, top_n=20):
    class_idx = list(model.classes_).index(class_label)
    coef = model.coef_[class_idx]
    
    top_pos_idx = np.argsort(coef)[-top_n:]
    
    return pd.DataFrame({
        "word": feature_names[top_pos_idx],
        "weight": coef[top_pos_idx]
    }).sort_values("weight", ascending=False)

top_words_5 = get_top_words(best_log_reg, feature_names, 5)
top_words_4 = get_top_words(best_log_reg, feature_names, 4)
top_words_3 = get_top_words(best_log_reg, feature_names, 3)
top_words_2 = get_top_words(best_log_reg, feature_names, 2)
top_words_1 = get_top_words(best_log_reg, feature_names, 1)

print("Top words for rating 5:\n", top_words_5)
print("\nTop words for rating 4:\n", top_words_4)
print("\nTop words for rating 3:\n", top_words_3)
print("\nTop words for rating 2:\n", top_words_2)
print("\nTop words for rating 1:\n", top_words_1)

Top words for rating 5:
                     word    weight
19           rev_amazing  3.322464
18              rev_love  2.690569
17            rev_smooth  2.500862
16              rev_soft  1.991288
15             rev_years  1.904275
14             rev_stuff  1.867592
13           rev_without  1.776363
12            rev_highly  1.769630
11          rev_favorite  1.748987
10  rev_highly recommend  1.692945
9             rev_helped  1.689984
8            rev_perfect  1.641838
7           rev_obsessed  1.563643
6            rev_already  1.547076
5               rev_skin  1.546992
4               rev_year  1.541926
3               rev_gone  1.537491
2               rev_best  1.476460
1         rev_complexion  1.447387
0               rev_must  1.425890

Top words for rating 4:
                 word    weight
19          rev_star  3.193948
18         rev_stars  2.471265
17      rev_one star  1.932608
16        rev_little  1.680110
15       rev_enjoyed  1.653044
14   rev_really like  1.6388

Logistic regression model performs well overall (71% accuracy), but lower macro f1 indicates weaker performance on less frequent rating classes, suggesting class imbalance impacts prediction quality.

## 2) Linear SVM

In [21]:
# Hyperparameter Tuning
for C in [0.01, 0.1, 1, 10]:
    for class_weight in [None, "balanced"]:
        svm = LinearSVC(
            C=C,
            class_weight=class_weight,
            max_iter=3000,
            random_state=42
        )
        
        svm.fit(X_train_review, y_train_review)
        preds = svm.predict(X_val_review)
        
        results.append({
            "C": C,
            "class_weight": class_weight,
            "val_accuracy": accuracy_score(y_val_review, preds),
            "val_f1_macro": f1_score(y_val_review, preds, average="macro")
        })

results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False)
print(results_df)

        C class_weight  val_accuracy  val_f1_macro
9    1.00     balanced         0.684      0.490264
10  10.00         None         0.670      0.484112
8    1.00         None         0.712      0.482754
7    0.10     balanced         0.694      0.478755
2    1.00         None         0.712      0.474184
3    1.00     balanced         0.712      0.474184
11  10.00     balanced         0.642      0.451744
6    0.10         None         0.708      0.428437
5    0.01     balanced         0.668      0.363114
1    0.10     balanced         0.664      0.305247
0    0.10         None         0.664      0.305247
4    0.01         None         0.648      0.246770


In [22]:
best_svm = LinearSVC(
    C=1,
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)

best_svm.fit(X_train_review, y_train_review)

test_preds = best_svm.predict(X_test_review)

print("Test Accuracy:", accuracy_score(y_test_review, test_preds))
print("\nTest Report:\n", classification_report(y_test_review, test_preds))

Test Accuracy: 0.702

Test Report:
               precision    recall  f1-score   support

           1       0.55      0.46      0.50        39
           2       0.23      0.32      0.27        19
           3       0.24      0.30      0.27        33
           4       0.46      0.43      0.44        74
           5       0.86      0.85      0.86       335

    accuracy                           0.70       500
   macro avg       0.47      0.47      0.47       500
weighted avg       0.71      0.70      0.71       500



In [30]:
top_words_5 = get_top_words(best_svm, feature_names, 5)
top_words_4 = get_top_words(best_svm, feature_names, 4)
top_words_3 = get_top_words(best_svm, feature_names, 3)
top_words_2 = get_top_words(best_svm, feature_names, 2)
top_words_1 = get_top_words(best_svm, feature_names, 1)

print("Top words for rating 5:\n", top_words_5)
print("Top words for rating 4:\n", top_words_4)
print("Top words for rating 3:\n", top_words_3)
print("Top words for rating 2:\n", top_words_2)
print("Top words for rating 1:\n", top_words_1)

Top words for rating 5:
                  word    weight
19        rev_amazing  1.900852
18       rev_favorite  1.576682
17     rev_complexion  1.509899
16          rev_stock  1.468079
15           rev_1010  1.434727
14          rev_years  1.408511
13         rev_damage  1.401413
12      rev_less oily  1.383948
11       rev_obsessed  1.358004
10           rev_gone  1.356333
9      rev_say enough  1.351203
8       rev_favourite  1.324469
7     rev_wear makeup  1.324062
6          rev_healed  1.310337
5          rev_smooth  1.308213
4        rev_improved  1.295191
3         rev_totally  1.238417
2          rev_hooked  1.227947
1         rev_healthy  1.206409
0   rev_product works  1.199123
Top words for rating 4:
                   word    weight
19            rev_star  3.482410
18        rev_one star  2.750075
17           rev_stars  2.288775
16            rev_mins  2.041427
15           rev_liked  1.809196
14      rev_like serum  1.764553
13         rev_enjoyed  1.751060
12        rev_

Linear SVM with class balancing achieved the better performance in predicting review ratings, particularly improving classification of minority rating classes. This demonstrates the effectiveness of margin-based models for high-dimensional TF-IDF text data. While accuracy dropped slightly, f1 score improved, which shows that the model is less biased towards 4-5 stars and better at predicting lower ratings.

## 3) SGDClassifier

In [27]:
# Hyperparameter Tuning
results = []

for loss in ["hinge", "log_loss"]:
    for alpha in [1e-5, 1e-4, 1e-3]:
        for class_weight in [None, "balanced"]:
            
            sgd = SGDClassifier(
                loss=loss,
                alpha=alpha,
                class_weight=class_weight,
                max_iter=1000,
                tol=1e-3,
                random_state=42,
                n_jobs=-1
            )
            
            sgd.fit(X_train_review, y_train_review)
            preds = sgd.predict(X_val_review)
            
            results.append({
                "loss": loss,
                "alpha": alpha,
                "class_weight": class_weight,
                "val_accuracy": accuracy_score(y_val_review, preds),
                "val_f1_macro": f1_score(y_val_review, preds, average="macro")
            })

results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False)
print(results_df)

        loss    alpha class_weight  val_accuracy  val_f1_macro
2      hinge  0.00010         None         0.728      0.503425
0      hinge  0.00001         None         0.706      0.487707
9   log_loss  0.00010     balanced         0.710      0.485897
3      hinge  0.00010     balanced         0.682      0.484796
6   log_loss  0.00001         None         0.698      0.472796
7   log_loss  0.00001     balanced         0.638      0.466983
1      hinge  0.00001     balanced         0.632      0.451013
8   log_loss  0.00010         None         0.708      0.427161
5      hinge  0.00100     balanced         0.670      0.414621
11  log_loss  0.00100     balanced         0.668      0.368453
4      hinge  0.00100         None         0.666      0.306957
10  log_loss  0.00100         None         0.646      0.250641


In [28]:
best_sgd = SGDClassifier(
    loss="hinge",
    alpha=1e-4,
    class_weight=None,
    max_iter=1000,
    tol=1e-3,
    random_state=42,
    n_jobs=-1
)

best_sgd.fit(X_train_review, y_train_review)

test_preds = best_sgd.predict(X_test_review)

print("Test Accuracy:", accuracy_score(y_test_review, test_preds))
print("\nTest Report:\n", classification_report(y_test_review, test_preds))

Test Accuracy: 0.744

Test Report:
               precision    recall  f1-score   support

           1       0.60      0.38      0.47        39
           2       0.17      0.21      0.19        19
           3       0.47      0.21      0.29        33
           4       0.51      0.28      0.37        74
           5       0.82      0.97      0.89       335

    accuracy                           0.74       500
   macro avg       0.51      0.41      0.44       500
weighted avg       0.71      0.74      0.71       500



In [31]:
top_words_5 = get_top_words(best_sgd, feature_names, 5)
top_words_4 = get_top_words(best_sgd, feature_names, 4)
top_words_3 = get_top_words(best_sgd, feature_names, 3)
top_words_2 = get_top_words(best_sgd, feature_names, 2)
top_words_1 = get_top_words(best_sgd, feature_names, 1)

print("Top words for rating 5:\n", top_words_5)
print("Top words for rating 4:\n", top_words_4)
print("Top words for rating 3:\n", top_words_3)
print("Top words for rating 2:\n", top_words_2)
print("Top words for rating 1:\n", top_words_1)

Top words for rating 5:
                word    weight
19      rev_amazing  2.669107
18         rev_gone  2.330862
17         rev_love  2.169535
16   rev_complexion  2.047775
15       rev_smooth  2.041323
14        rev_years  2.014343
13        rev_stuff  1.932445
12      rev_already  1.916163
11         rev_year  1.897898
10         rev_plus  1.877390
9   rev_much better  1.771205
8       rev_perfect  1.771185
7           rev_run  1.758314
6        rev_highly  1.738489
5         rev_magic  1.736308
4          rev_soft  1.729489
3       rev_without  1.720431
2          rev_1010  1.678453
1        rev_gifted  1.677971
0     rev_less oily  1.658549
Top words for rating 4:
                 word    weight
19          rev_star  5.406185
18         rev_stars  3.592904
17      rev_one star  3.400741
16  rev_strong scent  2.247836
15  rev_good product  2.131154
14  rev_nice product  2.086418
13       rev_enjoyed  2.039591
12  rev_giving stars  1.949129
11     rev_complaint  1.934835
10    rev_

The SGDClassifier achieves a test accuracy of 0.744, but this performance is largely driven by class imbalance, as 5-star reviews dominate the dataset (335 out of 500 samples). This imbalance inflates overall accuracy while masking weaker performance on minority classes, which is reflected in the much lower macro F1 score (0.44). The model tends to default toward predicting 5-star ratings, making it effective at identifying clearly positive reviews but less reliable at distinguishing between more nuanced or less frequent rating categories.

## Final Model

Although the SGDClassifier achieved higher accuracy, it was heavily biased toward predicting 5-star reviews due to class imbalance. The Linear SVM model provides a more balanced performance, with higher macro F1 and improved detection of minority classes, making it more suitable for capturing nuanced sentiment differences.

For 5-star reviews, top words like “amazing,” “favorite,” “obsessed,” and “product works” reflect strong satisfaction, but also emphasize visible results such as “improved,” “healed,” and “less oily.” This shows the model is learning that high ratings are driven not just by emotion, but by clear effectiveness.

For 4-star reviews, the model identifies balanced sentiment, with words like “liked” and “enjoyed” alongside negatives such as “pricy,” “downside,” and “strong scent.” This confirms that 4-star reviews are typically positive with trade-offs, which explains why they are harder to distinguish from 5-star but are better captured here.

For 3-star reviews, words like “okay,” “however,” and “unfortunately” indicate mixed or uncertain experiences, often with ongoing evaluation (“using weeks,” “research”). These reviews reflect partial satisfaction, making them inherently ambiguous and difficult to classify.

For 2-star and 1-star reviews, the model captures negative outcomes and unmet expectations. Clear signals like “irritated skin,” “didn’t find,” “worst,” and “don’t buy” indicate dissatisfaction, though some mixed phrases in 2-star reviews show that lower ratings can still include partial positives. Overall, the SVM better distinguishes subtle differences across ratings, especially between adjacent classes, leading to more balanced performance.

## Business Takeaway

For a startup launching on the Sephora marketplace, the key takeaway is: success depends on delivering clear, noticeable results and matching customer expectations. 5-star reviews are driven by visible performance (e.g., smoother skin, improved complexion), so your product and messaging should focus on one or two standout benefits customers can quickly see and describe.

The biggest opportunity is turning 4-star users into 5-star advocates. Most 4-star reviews are positive but include small issues like price, scent, or usability. Fixing these minor friction points (better packaging, clearer instructions, or expectation-setting) can significantly boost ratings and sales.

Finally, negative reviews highlight critical risks like irritation or lack of effectiveness. Minimizing these through strong formulation and testing is essential, especially early on. To stand out, your product needs a clear “wow factor”: not just good, but something customers actively rave about.